# Retrosynthese-Challenge: Product-SMILES → Reactant-SMILES

Dieses Notebook erstellt eine robuste Baseline für deine Aufgabe:

```text
[Reactants' SMILES] >> [Products' SMILES]
```

Ziel:

```text
Product SMILES → Reactant SMILES
```

Die Lösung orientiert sich am bereitgestellten Evaluationsskript `top1_accuracy.py`.

Wichtige Punkte aus der Evaluation:

- Die Submission hat **keinen Header**.
- Die Submission hat genau **eine Spalte**.
- Pro Testprodukt wird genau **eine Reactant-SMILES-Zeile** vorhergesagt.
- Reactants werden an `.` getrennt.
- Die Reihenfolge der Reactant-Moleküle ist egal.
- SMILES werden mit RDKit canonicalisiert.
- Verglichen wird als Set canonicalisierter Moleküle.

Die Baseline in diesem Notebook kombiniert:

1. **Exact Lookup**: Wenn ein Product im Training exakt vorkommt, wird der häufigste zugehörige Reactant genommen.
2. **Fingerprint Nearest Neighbor**: Falls kein Exact Match existiert, wird das ähnlichste Product aus dem Training gesucht.
3. **Fallback**: Falls RDKit ein Molekül nicht parsen kann, wird der häufigste Reactant aus dem Training verwendet.

## 0. Dateien und Setup

Lege dieses Notebook in denselben Ordner wie:

```text
data_train.csv
product_smiles_test.csv
sample_submission.csv
top1_accuracy.py
```

Falls RDKit fehlt, installiere es zum Beispiel mit:

```bash
conda install -c conda-forge rdkit
```

oder, falls Conda nicht verfügbar ist:

```bash
pip install rdkit-pypi
```

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import random
import math
import os

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "data_train.csv"
TEST_PATH = DATA_DIR / "product_smiles_test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

print("Working directory:", Path.cwd())
print("Train exists:", TRAIN_PATH.exists())
print("Test exists:", TEST_PATH.exists())
print("Sample submission exists:", SAMPLE_SUB_PATH.exists())

## 1. Hilfsfunktionen für Canonicalisierung

Diese Funktionen sind absichtlich ähnlich zum Evaluationsskript gebaut.

Das Evaluationsskript macht im Kern:

```python
Chem.MolFromSmiles(chem)
Chem.MolToSmiles(m)
```

und vergleicht danach Sets.

Deshalb canonicalisieren wir:

- einzelne Moleküle
- Molekül-Sets, die mit `.` getrennt sind
- komplette Reaktionszeilen im Format `Reactants >> Products`

In [ ]:
def canonical_mol(smiles: str):
    """
    Canonicalisiert ein einzelnes Molekül-SMILES mit RDKit.
    Gibt None zurück, falls RDKit das Molekül nicht parsen kann.
    """
    if smiles is None:
        return None

    smiles = str(smiles).strip()
    if smiles == "":
        return None

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    return Chem.MolToSmiles(mol)


def canonical_set(smiles: str) -> str:
    """
    Canonicalisiert eine mit '.' getrennte Liste von Molekülen.

    Wichtig:
    - Ungültige Moleküle werden ignoriert, genau wie im Evaluationsskript.
    - Doppelte Moleküle werden entfernt, weil das Evaluationsskript Sets verwendet.
    - Die Moleküle werden sortiert, damit die Darstellung stabil ist.
    """
    if smiles is None:
        return ""

    parts = str(smiles).strip().split(".")
    canon = []

    for part in parts:
        c = canonical_mol(part)
        if c is not None:
            canon.append(c)

    return ".".join(sorted(set(canon)))


def parse_reaction_line(line: str):
    """
    Parst eine Reaktionszeile:

    Reactants >> Products

    Rückgabe:
    canonical_reactants, canonical_products
    """
    line = str(line).strip()
    if ">>" not in line:
        return None, None

    reactants, products = line.split(">>", 1)
    reactants = canonical_set(reactants.strip())
    products = canonical_set(products.strip())

    if reactants == "" or products == "":
        return None, None

    return reactants, products


def evaluation_equal(pred: str, true: str) -> bool:
    """
    Entspricht der Vergleichslogik aus top1_accuracy.py.
    """
    return set(canonical_set(pred).split(".")) == set(canonical_set(true).split("."))

## 2. Daten laden

`data_train.csv` enthält eine Reaktion pro Zeile.

Beispiel:

```text
C/C(=C(/C(=O)N)\C)/O >> NC(=O)O.CC#CC
```

Wir lesen die Datei bewusst zeilenweise, weil die Trainingsdatei offenbar keinen Header hat.

In [ ]:
def read_single_column_text_file(path: Path):
    """
    Liest eine Datei als Liste nicht-leerer Zeilen.
    Funktioniert für headerlose einspaltige CSV-/Textdateien.
    """
    with open(path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip() != ""]
    return lines


train_lines = read_single_column_text_file(TRAIN_PATH)
test_products_raw = read_single_column_text_file(TEST_PATH)

print("Number of train reactions:", len(train_lines))
print("Number of test products:", len(test_products_raw))
print()
print("First 4 train lines:")
for x in train_lines[:4]:
    print(x)
print()
print("First 4 test products:")
for x in test_products_raw[:4]:
    print(x)

## 3. Trainingsdaten in Product → Reactants umwandeln

Das Modell soll aus Products Reactants vorhersagen.

Daher drehen wir die Reaktionsrichtung:

```text
Reactants >> Products
```

wird zu:

```text
Products → Reactants
```

In [ ]:
pairs = []

invalid_lines = 0

for line in train_lines:
    reactants, products = parse_reaction_line(line)
    if reactants is None or products is None:
        invalid_lines += 1
        continue
    pairs.append({
        "product": products,
        "reactants": reactants,
        "raw_line": line,
    })

df = pd.DataFrame(pairs)

print("Valid parsed reactions:", len(df))
print("Invalid lines skipped:", invalid_lines)
df.head()

In [ ]:
print("Unique canonical products:", df["product"].nunique())
print("Unique canonical reactant sets:", df["reactants"].nunique())

duplicate_product_counts = df["product"].value_counts()
print("Products appearing more than once:", (duplicate_product_counts > 1).sum())

df.head(10)

## 4. Lokale Top-1-Accuracy-Funktion

Diese Funktion entspricht der bereitgestellten Evaluation.

Damit können wir auf einem lokalen Validierungssplit testen.

In [ ]:
def calculate_top1_accuracy_from_lists(predictions, true_answers):
    assert len(predictions) == len(true_answers), "Predictions and targets must have same length."

    correct = 0
    total = 0

    for pred, true in zip(predictions, true_answers):
        if pred is None or true is None:
            continue

        pred = str(pred).strip()
        true = str(true).strip()

        if pred == "" or true == "":
            continue

        true_set = set(canonical_set(true).split("."))
        pred_set = set(canonical_set(pred).split("."))

        if true_set == pred_set:
            correct += 1

        total += 1

    if total == 0:
        return 0.0

    return correct / total


# Mini-Test: Reihenfolge der Moleküle sollte egal sein
assert calculate_top1_accuracy_from_lists(["A.B"], ["B.A"]) == 0.0  # A und B sind keine validen SMILES
assert calculate_top1_accuracy_from_lists(["CC.O"], ["O.CC"]) == 1.0

print("Evaluation helper works.")

## 5. Modell: Exact Lookup + Fingerprint Nearest Neighbor

Die Strategie:

1. Canonicalisiere das Test-Product.
2. Falls dieses Product im Training vorkommt:
   - gib den häufigsten Reactant zu diesem Product zurück.
3. Falls nicht:
   - berechne einen Morgan-Fingerprint für das Test-Product.
   - suche das ähnlichste Product im Training per Tanimoto Similarity.
   - gib dessen Reactants zurück.
4. Falls RDKit scheitert:
   - gib den häufigsten Reactant im Training zurück.

In [ ]:
class RetrosynthesisBaseline:
    def __init__(self, radius=2, n_bits=2048):
        self.radius = radius
        self.n_bits = n_bits

        self.product_to_reactants_counter = defaultdict(Counter)
        self.product_to_best_reactants = {}
        self.most_common_reactants = None

        self.train_products = []
        self.train_reactants = []
        self.train_fps = []

    def _fp(self, smiles: str):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=self.radius,
            nBits=self.n_bits
        )

    def fit(self, train_df: pd.DataFrame):
        """
        Erwartet ein DataFrame mit Spalten:
        - product
        - reactants
        """
        self.product_to_reactants_counter = defaultdict(Counter)

        for product, reactants in zip(train_df["product"], train_df["reactants"]):
            self.product_to_reactants_counter[product][reactants] += 1

        self.product_to_best_reactants = {
            product: counter.most_common(1)[0][0]
            for product, counter in self.product_to_reactants_counter.items()
        }

        self.most_common_reactants = Counter(train_df["reactants"]).most_common(1)[0][0]

        self.train_products = []
        self.train_reactants = []
        self.train_fps = []

        # Für kNN nehmen wir pro Produkt am besten nur den häufigsten Reactant.
        # Dadurch vermeiden wir unnötige Duplikate.
        for product, reactants in self.product_to_best_reactants.items():
            fp = self._fp(product)
            if fp is not None:
                self.train_products.append(product)
                self.train_reactants.append(reactants)
                self.train_fps.append(fp)

        print("Fitted model")
        print("Unique products in lookup:", len(self.product_to_best_reactants))
        print("Products with valid fingerprints:", len(self.train_fps))
        print("Most common reactants:", self.most_common_reactants)

        return self

    def predict_one(self, product_smiles: str):
        product = canonical_set(product_smiles)

        # 1. Exact product lookup
        if product in self.product_to_best_reactants:
            return self.product_to_best_reactants[product]

        # 2. Fingerprint nearest neighbor
        fp = self._fp(product)
        if fp is None or len(self.train_fps) == 0:
            return self.most_common_reactants

        sims = DataStructs.BulkTanimotoSimilarity(fp, self.train_fps)
        best_idx = int(np.argmax(sims))

        return self.train_reactants[best_idx]

    def predict(self, product_smiles_list):
        return [self.predict_one(x) for x in product_smiles_list]

## 6. Random Validation Split

Dieser Split ist einfach, kann aber optimistisch sein.

Warum?

Wenn dasselbe Product sowohl im Training als auch im Validierungsset vorkommt, kann Exact Lookup die Antwort direkt finden.

In [ ]:
def random_train_valid_split(df, valid_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    indices = np.arange(len(df))
    rng.shuffle(indices)

    n_valid = int(len(df) * valid_frac)
    valid_idx = indices[:n_valid]
    train_idx = indices[n_valid:]

    train_df = df.iloc[train_idx].reset_index(drop=True)
    valid_df = df.iloc[valid_idx].reset_index(drop=True)

    return train_df, valid_df


train_random_df, valid_random_df = random_train_valid_split(df, valid_frac=0.2, seed=SEED)

print("Random train size:", len(train_random_df))
print("Random valid size:", len(valid_random_df))

model_random = RetrosynthesisBaseline(radius=2, n_bits=2048)
model_random.fit(train_random_df)

valid_random_preds = model_random.predict(valid_random_df["product"].tolist())
random_acc = calculate_top1_accuracy_from_lists(valid_random_preds, valid_random_df["reactants"].tolist())

print("Random validation Top-1 accuracy:", random_acc)

## 7. Product-disjoint Validation Split

Dieser Split ist strenger.

Hier wird sichergestellt:

```text
Kein Product aus dem Validierungsset kommt im Trainingssplit vor.
```

Damit testest du eher, ob der Fingerprint-kNN-Fallback generalisieren kann.

In [ ]:
def product_disjoint_split(df, valid_product_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)

    unique_products = np.array(df["product"].unique())
    rng.shuffle(unique_products)

    n_valid_products = int(len(unique_products) * valid_product_frac)
    valid_products = set(unique_products[:n_valid_products])

    valid_mask = df["product"].isin(valid_products)

    train_df = df.loc[~valid_mask].reset_index(drop=True)
    valid_df = df.loc[valid_mask].reset_index(drop=True)

    return train_df, valid_df


train_disjoint_df, valid_disjoint_df = product_disjoint_split(df, valid_product_frac=0.2, seed=SEED)

print("Disjoint train size:", len(train_disjoint_df))
print("Disjoint valid size:", len(valid_disjoint_df))
print("Product overlap:", len(set(train_disjoint_df["product"]) & set(valid_disjoint_df["product"])))

model_disjoint = RetrosynthesisBaseline(radius=2, n_bits=2048)
model_disjoint.fit(train_disjoint_df)

valid_disjoint_preds = model_disjoint.predict(valid_disjoint_df["product"].tolist())
disjoint_acc = calculate_top1_accuracy_from_lists(valid_disjoint_preds, valid_disjoint_df["reactants"].tolist())

print("Product-disjoint validation Top-1 accuracy:", disjoint_acc)

## 8. Fehleranalyse

Wir schauen uns einige Validierungsbeispiele an, bei denen die Vorhersage falsch war.

Das hilft, später bessere Regeln oder ein Seq2Seq-Modell zu entwickeln.

In [ ]:
analysis_df = valid_disjoint_df.copy()
analysis_df["prediction"] = valid_disjoint_preds
analysis_df["correct"] = [
    evaluation_equal(pred, true)
    for pred, true in zip(analysis_df["prediction"], analysis_df["reactants"])
]

print("Correct:", analysis_df["correct"].sum())
print("Total:", len(analysis_df))
print("Accuracy:", analysis_df["correct"].mean())

wrong_examples = analysis_df.loc[~analysis_df["correct"], ["product", "reactants", "prediction"]].head(20)
wrong_examples

## 9. Optional: Verschiedene Fingerprint-Einstellungen testen

Hier testen wir ein paar Morgan-Fingerprint-Konfigurationen auf dem product-disjoint Split.

Das kann helfen, eine bessere Einstellung zu finden.

In [ ]:
configs = [
    {"radius": 1, "n_bits": 1024},
    {"radius": 2, "n_bits": 1024},
    {"radius": 2, "n_bits": 2048},
    {"radius": 3, "n_bits": 2048},
    {"radius": 3, "n_bits": 4096},
]

results = []

for cfg in configs:
    print("Testing:", cfg)
    m = RetrosynthesisBaseline(radius=cfg["radius"], n_bits=cfg["n_bits"])
    m.fit(train_disjoint_df)
    preds = m.predict(valid_disjoint_df["product"].tolist())
    acc = calculate_top1_accuracy_from_lists(preds, valid_disjoint_df["reactants"].tolist())
    results.append({**cfg, "top1_accuracy": acc})

results_df = pd.DataFrame(results).sort_values("top1_accuracy", ascending=False)
results_df

## 10. Finales Modell auf allen Trainingsdaten trainieren

Nachdem die Baseline lokal getestet wurde, trainieren wir das finale Modell auf allen Trainingsdaten.

In [ ]:
# Nimm entweder die beste Konfiguration aus dem Test oben
# oder setze manuell eine robuste Default-Konfiguration.
best_radius = int(results_df.iloc[0]["radius"]) if "results_df" in globals() and len(results_df) > 0 else 2
best_n_bits = int(results_df.iloc[0]["n_bits"]) if "results_df" in globals() and len(results_df) > 0 else 2048

print("Using final config:")
print("radius:", best_radius)
print("n_bits:", best_n_bits)

final_model = RetrosynthesisBaseline(radius=best_radius, n_bits=best_n_bits)
final_model.fit(df)

## 11. Testdaten vorhersagen

`product_smiles_test.csv` enthält nur Product-SMILES.

Wir erzeugen eine Prediction pro Zeile.

In [ ]:
test_products_canonical = [canonical_set(x) for x in test_products_raw]
test_predictions = final_model.predict(test_products_raw)

print("Number of predictions:", len(test_predictions))
print("First 10 predictions:")
for p in test_predictions[:10]:
    print(p)

## 12. Submission speichern

Die Submission muss laut `sample_submission.csv` und Evaluationsskript so aussehen:

- kein Header
- genau eine Spalte
- genau so viele Zeilen wie `product_smiles_test.csv`

In [ ]:
submission_path = Path("submission.csv")

pd.Series(test_predictions).to_csv(
    submission_path,
    index=False,
    header=False
)

print("Saved:", submission_path.resolve())
print("Rows:", len(test_predictions))

# Sanity check: wieder einlesen
sub_check = pd.read_csv(submission_path, header=None, names=["prediction"])
print(sub_check.shape)
sub_check.head()

## 13. Optionaler Check gegen `sample_submission.csv`

Dieser Check prüft nur, ob die Anzahl der Zeilen übereinstimmt.

Die Beispielsubmission enthält natürlich nicht die richtigen Antworten.

In [ ]:
if SAMPLE_SUB_PATH.exists():
    sample_sub = pd.read_csv(SAMPLE_SUB_PATH, header=None, names=["prediction"])
    print("Sample submission rows:", len(sample_sub))
    print("Our submission rows:", len(test_predictions))

    if len(sample_sub) == len(test_predictions):
        print("OK: Row count matches sample_submission.csv")
    else:
        print("WARNING: Row count does not match sample_submission.csv")
else:
    print("sample_submission.csv not found, skipping check.")

## 14. Optional: Lokale Evaluation mit Target-Datei

Falls du irgendwann eine lokale Target-Datei hast, zum Beispiel:

```text
valid_targets.csv
```

mit einer Spalte ohne Header, kannst du deine Submission so testen:

```python
local_targets = pd.read_csv("valid_targets.csv", header=None, names=["true_reactants"])
local_preds = pd.read_csv("submission.csv", header=None, names=["prediction"])

acc = calculate_top1_accuracy_from_lists(
    local_preds["prediction"].tolist(),
    local_targets["true_reactants"].tolist()
)

print(acc)
```

In [ ]:
# Optionaler Block, standardmäßig deaktiviert.
# Passe TARGET_PATH an, falls du eine lokale Ground-Truth-Datei hast.

TARGET_PATH = Path("valid_targets.csv")

if TARGET_PATH.exists():
    local_targets = pd.read_csv(TARGET_PATH, header=None, names=["true_reactants"])
    local_preds = pd.read_csv("submission.csv", header=None, names=["prediction"])

    acc = calculate_top1_accuracy_from_lists(
        local_preds["prediction"].tolist(),
        local_targets["true_reactants"].tolist()
    )

    print("Local target Top-1 accuracy:", acc)
else:
    print("No local target file found. Skipping.")

## 15. Nächste Verbesserungen

Falls diese Baseline nicht reicht, wären die nächsten sinnvollen Schritte:

1. **Ensemble mehrerer kNN-Modelle**
   - Morgan radius 2
   - Morgan radius 3
   - MACCS Keys
   - RDKit Fingerprint

2. **Top-k ähnliche Products betrachten**
   - nicht nur den besten Nachbarn nehmen
   - sondern die häufigsten Reactants unter den Top-k Nachbarn wählen

3. **SMILES-Seq2Seq-Modell**
   - Input: Product SMILES
   - Output: Reactant SMILES
   - zum Beispiel ein kleiner Transformer
   - danach nur valide RDKit-SMILES behalten

4. **Hybridmodell**
   - Exact Lookup zuerst
   - danach kNN
   - danach Seq2Seq
   - danach häufigster Reactant als Fallback

Für diese Challenge ist die hier gebaute Baseline ein guter erster Schritt, weil sie schnell, stabil und exakt kompatibel mit der Evaluation ist.